In [ ]:
"""
compare_3d_roundnesses.ipynb
Author: Henry Yuan
Description: Code for plots in Figure 4 of paper. Plots values of Wadell roundness and
new standard deviation of curvature of synthetic grains versus mesh vertex count,
and plots local ranges of the metric
"""

# imports
import numpy as np
import sys
import matplotlib.pyplot as plt
sys.path.append('../../src')
import pandas as pd
from format_plots import format_plots

CR, CS, CE, CL, CRD, CED, CWAD, CSTD = format_plots() # format plots and collect colors

In [ ]:
# load roundness data for synthetic grains
data_file = '../../data/synthetic/random_synthetics/synthetic_stats.csv'
synthetic_stats_df = pd.read_csv(data_file)

# unpack into original roundness stage (round_stages), vertex count (nvs), and desired roundness metrics
metric_names = ['std_curv','wadell']
round_stages = synthetic_stats_df['roundness']
nvs = synthetic_stats_df['vertex_count']
std_curv = synthetic_stats_df['std_curv']
wadell = synthetic_stats_df['wadell']

In [ ]:
# function for more plot formatting, specific to this figure
def set_plot_layout():
    fig, axs = plt.subplots(3,1,figsize=(3.46,6),sharex=True)
    axs = axs.flatten()
    axs[2].set_xlabel('Mesh vertex count')

    for ax in axs:
        ax.set_xlim(500,10500)   # axis starts at 1000 (lowest x)
        ax.set_xticks(np.arange(500,10501,500))  # explicit ticks, first one is 1000
        ticks = ax.get_xticks()
        ax.set_xticklabels([f'{int(t)}' if (i % 4 == 1) else '' for i, t in enumerate(ticks)])
        ax.axvline(1000,ls='--',lw=1,color='black',alpha=0.5)
        ax.axvline(5000,ls='--',lw=1,color='black')

    normal_length = plt.rcParams["xtick.major.size"]  # default 3.5
    half_length = normal_length / 2

    # Draw the figure once so tick objects are fully created
    fig.canvas.draw()

    for ax in axs:
        for tick in ax.xaxis.get_major_ticks():
            value = tick.get_loc()
            if np.isclose(value % 1000, 0) or np.isclose(value % 1000, 1000):
                tick.tick1line.set_markersize(normal_length)  # bottom tick
                tick.tick2line.set_markersize(normal_length)  # top tick (if visible)
            else:
                tick.tick1line.set_markersize(half_length)
                tick.tick2line.set_markersize(half_length)

    return fig, axs

In [ ]:
# PLOT ROUNDNESS VS RESOLUTION AND RANGE EVOLUTION

# set plot layout
fig, axs = set_plot_layout()
alpha = 0.5

# define local bins over which means will be calculated and plotted
group_size=1000
group_ranges = np.array([[i,i+1000] for i in np.arange(1000,10000,group_size)])
group_ranges[-1,1] = 10001 # include 10000 in last group
n_groups = group_ranges.shape[0]
group_midpoints = np.mean(group_ranges,axis=1)

# make arrays to save mean of round and euhedral values for each bin for range normalization
# in panel 3 of this figure and Figure 5
wad_extrema_means = np.zeros((2,n_groups))
std_extrema_means = np.zeros((2,n_groups))

cs = [CE,CS,CR]

metric_labels = [r'$R_{W,\kappa}$',r'$R_{\sigma,\kappa}$']
rs = [0,0.5,1]
r_labels = ['euhedral','part round','round']

# label y-axes
for i,m in enumerate(metric_labels):
    axs[i].set_ylabel(m)

# loop through all roundness stages
for i,r in enumerate(rs):
    # subscript _i means "specific for this roundness stage"

    # get bool index list of all grains with roundness = r
    bool_list_i = (round_stages == r)
    # index into arrays to get subset of entire arrays for this specific r value
    nvs_i = nvs[bool_list_i]
    wadell_i = wadell[bool_list_i]
    std_curv_i = std_curv[bool_list_i]

    # scatter plot of all synthetics for wadell and std_curv
    axs[0].scatter(nvs_i, wadell_i,color=cs[i],label=r_labels[i],s=1,alpha=alpha)
    axs[1].scatter(nvs_i, std_curv_i,color=cs[i],label=r_labels[i],s=1,alpha=alpha)

    # now calculate locally-binned means with errorbars using bins defined above
    # skip if semi-round
    if i == 1:
        continue
    
    # set up arrays for locally-binned means and standard deviations 
    wad_means_i = np.zeros(n_groups)
    wad_stds_i = np.zeros(n_groups)
    std_means_i = np.zeros(n_groups)
    std_stds_i = np.zeros(n_groups)
    for j, rg in enumerate(group_ranges):
        # subscript j means "specific to this range"
        gmin,gmax = rg
        group_bool_inds_j = np.logical_and(nvs_i >= gmin, nvs_i < gmax)

        wadell_ij = wadell_i[group_bool_inds_j]
        std_curv_ij = std_curv_i[group_bool_inds_j]

        wad_means_i[j] = wadell_ij.mean()
        wad_stds_i[j] = wadell_ij.std()
        std_means_i[j] = std_curv_ij.mean()
        std_stds_i[j] = std_curv_ij.std()

    # save means for range calculation later
    # i // 2 maps indices 0, 2 to 0, 1 since we are skipping semi-roung subset (i = 1)
    wad_extrema_means[i//2] = wad_means_i
    std_extrema_means[i//2] = std_means_i

    # set color and marker
    if i == 0:
        c = CED
        mk = 's'
    if i == 2:
        c = CRD
        mk = 'o'

    # plot means with errorbars
    axs[0].errorbar(group_midpoints, wad_means_i,yerr=wad_stds_i,color=CL,marker=mk,markersize=4, markerfacecolor=c,mec=CL,mew=0.5,lw=1,capsize=2)
    axs[1].errorbar(group_midpoints, std_means_i,yerr=std_stds_i,color=CL,marker=mk,markersize=4, markerfacecolor=c,mec=CL,mew=0.5,lw=1,capsize=2)
    
axs[0].legend()

# reset ylims; manually chosen values
ylims = axs[0].get_ylim()
axs[0].set_ylim((ylims[0],0.95))
ylims = axs[1].get_ylim()
axs[1].set_ylim((ylims[0],6.75))

# *** PLOT RANGES AS FRACTION OF MAX RANGE (or range at last grouping) IN 3RD PANEL ***

colors = [CWAD, CSTD]
mks = ['^','D']

wad_ranges = np.abs(wad_extrema_means[0] - wad_extrema_means[1])
std_ranges = np.abs(std_extrema_means[0] - std_extrema_means[1])
ranges_concat = [wad_ranges, std_ranges]

for i in range(2):
    axs[2].plot(group_midpoints, ranges_concat[i] / ranges_concat[i][-1], marker=mks[i],markersize=4, lw=1,color=colors[i],label=metric_labels[i])

axs[2].legend()
axs[2].set_ylabel('Normalized metric range')

plt.tight_layout()
plt.savefig('figure4.pdf')

In [ ]:
# print min and max values of wad_extrema_means and std_extrema_means to get range for normalizing DZ data
# in Figure 5
# for euhedral, print MIN of Wadell and MAX of std_curv (inveted scale)
print(f'Wadell experimental MINIMUM = {wad_extrema_means[0].min()}')
print(f'std_curv experimental MAXIMUM = {std_extrema_means[0].max()}')

# for round, print MAX of Wadell and MIN of std_curv (inveted scale)
print(f'Wadell experimental MAXIMUM = {wad_extrema_means[1].max()}')
print(f'std_curv experimental MINIMUM = {std_extrema_means[1].min()}')

